# eur_r2 — ancestry gate

Refits PCA on the round-1 participants, projects 1000G into that space, and keeps
those closest to the CEU + GBR centroid — the anchor Kemper et al. used.

Why an anchor rather than the participants' own centroid: a self-centred gate has
no external reference, so it drifts with whoever happens to be in the round-1 set.
The anchor fixes the centre and the relative PC weighting to a population that
exists independently of this cohort. Both alternatives are built at the end for
comparison.

**Runs:** QC and pruning on Batch (`n1-highmem-16`), PCA on Batch
(`n1-highmem-32`, the long step), scoring and everything after on this VM.

**Reads:** round-1 keep list from the ancestry survey.
**Writes:** `R/01_ancestry/round2/` — keep list, panels, PCA, figures, provenance.

## setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.expanduser("~/aou-covariance/runs/_lib"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aoucov import Run, QC_MACHINE, QC_MEM_MB, PCA_MACHINE, PCA_MEM_MB
from aoucov import batch, checks, env, gate, plink, plots, provenance, refs

run = Run("eur_r2")
OUT, OUT_GS = run.out("01_ancestry", "round2"), run.out_gs("01_ancestry", "round2")
LOCAL = run.scratch("round2")

ROUND1 = f"{run.b}/analyses/ancestry_filtering/keep/EUR_99pct_keep_ids.txt"
ROUND1_GS = f"{run.b_gs}/analyses/ancestry_filtering/keep/EUR_99pct_keep_ids.txt"

N_PCS_FIT = 20
K_PCS = 4          # PCs the gate uses; confirm against anchor separation below
COVERAGE = 0.90    # fraction of anchor samples inside the gate

refs.assert_inputs()
assert os.path.isfile(ROUND1), f"{ROUND1} — run the ancestry survey first"
print(f"round 1: {sum(1 for _ in open(ROUND1)):,} participants")
print(f"output:  {OUT}")

## install dsub

In [ ]:
env.ensure_dsub()

## QC, HM3 filter, prune — Batch

Keeps the round-1 samples, reapplies MAF/HWE/missingness in that cohort, keeps
variants agreeing with 1000G on ID and both alleles, then prunes hard (r²=0.05)
for the gate PCA.

`r1_qc` is exported to the bucket, not just the worker's scratch — the covariate
PCA reads it.

In [ ]:
LD_REGIONS_GS = f"{OUT_GS}/high_ld_regions.txt"
refs.write_ld_regions(f"{OUT}/high_ld_regions.txt", refs.HIGH_LD_WITH_PEAKS)

qc_job = batch.submit(run, "r2-qc", """
set -eo pipefail
chmod +x "$PLINK_BIN"
Q="${OUT_DIR}/r1_qc"

"$PLINK_BIN" --pgen "$PANEL_PGEN" --pvar "$PANEL_PVAR" --psam "$PANEL_PSAM" \\
  --keep "$KEEP_PATH" --nonfounders \\
  --maf 0.01 --hwe 1e-6 0 keep-fewhet --geno 0.05 --max-alleles 2 --rm-dup exclude-all \\
  --threads "$VCPUS" --memory "$MEM_MB" --make-pgen --out "$Q"

grep -v '^##' "${Q}.pvar" | awk 'NR>1 {print $3, $4, $5}' | LC_ALL=C sort > /tmp/lhs
awk 'NR>1 {print $2, $3, $4}' "$KG_ACOUNT" | LC_ALL=C sort > /tmp/kg
LC_ALL=C comm -12 /tmp/lhs /tmp/kg | awk '{print $1}' > "${OUT_DIR}/hm3_agreeing.ids"
echo "HM3-agreeing: $(wc -l < "${OUT_DIR}/hm3_agreeing.ids")"

"$PLINK_BIN" --pfile "$Q" --extract "${OUT_DIR}/hm3_agreeing.ids" \\
  --exclude bed1 "$LD_REGIONS" --nonfounders --indep-pairwise 1000kb 1 0.05 \\
  --threads "$VCPUS" --memory "$MEM_MB" --out "${OUT_DIR}/prune"

"$PLINK_BIN" --pfile "$Q" --extract "${OUT_DIR}/prune.prune.in" \\
  --threads "$VCPUS" --memory "$MEM_MB" --make-pgen --out "${OUT_DIR}/pca_input"
echo "PCA variants: $(wc -l < "${OUT_DIR}/prune.prune.in")"
""",
    machine_type=QC_MACHINE, mem_mb=QC_MEM_MB,
    inputs={
        "PANEL_PGEN": f"{run.panel_gs}.pgen",
        "PANEL_PVAR": f"{run.panel_gs}.pvar",
        "PANEL_PSAM": f"{run.panel_gs}.psam",
        "PLINK_BIN": run.plink2_gs,
        "KEEP_PATH": ROUND1_GS,
        "LD_REGIONS": LD_REGIONS_GS,
        "KG_ACOUNT": refs.KG_ACOUNT_GS,
    },
    outputs={"OUT_DIR": f"{OUT_GS}/panels"},
)

In [ ]:
batch.watch(run, qc_job)

## PCA — Batch

In [ ]:
pca_job = batch.submit(run, "r2-pca", f"""
set -eo pipefail
chmod +x "$PLINK_BIN"
"$PLINK_BIN" --pgen "$PCA_PGEN" --pvar "$PCA_PVAR" --psam "$PCA_PSAM" \\
  --nonfounders --freq counts \\
  --pca approx {N_PCS_FIT} allele-wts \\
  --threads "$VCPUS" --memory "$MEM_MB" --out "${{OUT_DIR}}/round2_pca"
echo done
""",
    machine_type=PCA_MACHINE, mem_mb=PCA_MEM_MB,
    inputs={
        "PLINK_BIN": run.plink2_gs,
        "PCA_PGEN": f"{OUT_GS}/panels/pca_input.pgen",
        "PCA_PVAR": f"{OUT_GS}/panels/pca_input.pvar",
        "PCA_PSAM": f"{OUT_GS}/panels/pca_input.psam",
    },
    outputs={"OUT_DIR": f"{OUT_GS}/pca"},
)

In [ ]:
batch.watch(run, pca_job)

## score everyone through the same loadings

Participants are re-scored through their own loadings rather than read from the
eigenvec — a `--pca` eigenvector and a `--score` projection do not land on the
same coordinates, and the gate needs both sets on identical axes.

In [ ]:
env.ensure_plink2()
env.sh(f"""
gsutil -m cp \\
  "{OUT_GS}/pca/round2_pca.eigenval" \\
  "{OUT_GS}/pca/round2_pca.eigenvec.allele" \\
  "{OUT_GS}/pca/round2_pca.acount" \\
  "{OUT_GS}/panels/pca_input."* \\
  "{OUT_GS}/panels/prune.prune.in" \\
  "{LOCAL}/"
""")

PCA = f"{LOCAL}/round2_pca"
W, FREQ = f"{PCA}.eigenvec.allele", f"{PCA}.acount"
PANEL = f"{LOCAL}/pca_input"

kg_ids = refs.shared_variants(f"{PANEL}.pvar", f"{LOCAL}/kg_project.ids")

plink.score(W, f"{LOCAL}/kg_in_r2", bfile=refs.KG_BFILE, freq=FREQ,
            extract=kg_ids, n_pcs=N_PCS_FIT)
plink.score(W, f"{LOCAL}/part_in_r2", pfile=PANEL, freq=FREQ, n_pcs=N_PCS_FIT)

kg = plink.read_scores(f"{LOCAL}/kg_in_r2.sscore", "sample", N_PCS_FIT).merge(
    refs.panel(), on="sample", how="left")
part = plink.read_scores(f"{LOCAL}/part_in_r2.sscore", "person_id", N_PCS_FIT)
print(f"{len(part):,} participants, {len(kg):,} 1000G samples")

## pick K

`anchor_vs_other_EUR` is the gap between CEU+GBR and the other European
populations, in participant SDs. Use the PCs where it is large; a PC that does
not separate the anchor adds noise to the Mahalanobis distance, not selectivity.

In [ ]:
_, pct = plink.read_eigenval(PCA)
sep = plots.anchor_separation(kg, part, N_PCS_FIT)
print(pd.DataFrame({"PC": plink.pc_names(N_PCS_FIT),
                    "pct_var": pct.round(2),
                    "anchor_vs_other_EUR": sep.round(2)}).to_string(index=False))

plots.scree(PCA, f"{OUT}/scree.png", title="eur_r2 round 2")

## loadings

In [ ]:
L = plots.loadings_by_position(PCA, f"{OUT}/loadings.png", n_show=K_PCS,
                               title="eur_r2 round 2 — PCA loadings")
peaks = checks.ld_peaks(L, n_pcs=K_PCS)

If `peaks` is non-empty, add them to `refs.HIGH_LD_WITH_PEAKS` and rerun from the
QC job.

## the gate

In [ ]:
anchor = kg[kg["pop"].isin(refs.ANCHOR_POPS)]
g = gate.fit(anchor, K_PCS, COVERAGE)

keep = g.keep(part)
KEEP_PATH = f"{OUT}/eur_r2_keep_ids.txt"
part.loc[keep, "person_id"].to_csv(KEEP_PATH, index=False, header=False)

n_variants = sum(1 for _ in open(f"{LOCAL}/prune.prune.in"))
provenance.write(OUT, "round2", {
    "pca": f"fit on the round-1 set, plink2 --pca approx {N_PCS_FIT} allele-wts",
    "variants": f"{n_variants} pruned HM3 sites (r2=0.05)",
    "anchor": f"1000G {'+'.join(refs.ANCHOR_POPS)} projected into that space",
    "pcs": f"1-{K_PCS}",
    "metric": "Mahalanobis under the anchor's own covariance",
    "threshold": f"{COVERAGE:.0%} quantile of anchor distances = {g.threshold:.6f}",
    "kept": int(keep.sum()),
})
provenance.result_line("gate", n_in=len(part), n_variants=n_variants,
                       k_pcs=K_PCS, radius=round(g.threshold, 4),
                       n_kept=int(keep.sum()))

In [ ]:
plots.pc_pairs(part, kg, f"{OUT}/round2_pcs.png", gate=g, mask=keep,
               title="eur_r2 — kept (blue) within the round-1 set (grey), "
                     "1000G Europeans (crosses)")

## comparison gates

The participants' own centroid, and the Kemper direction (PCA fit on the 1000G
Europeans, participants projected in). Neither replaces the gate above; they
bound how much the anchor choice actually matters.

In [ ]:
g_self = gate.self_gate(part, K_PCS, COVERAGE)
keep_self = g_self.keep(part)

refs.eur_keep(f"{LOCAL}/kg_eur.keep")
# plink.pca() takes a pfile; 1000G is a bfile here, so this one is spelled out.
env.sh(f"""
plink2 --bfile "{refs.KG_BFILE}" --keep "{LOCAL}/kg_eur.keep" \\
  --extract "{kg_ids}" --nonfounders --freq counts \\
  --pca {N_PCS_FIT} allele-wts --threads $(nproc) --out "{LOCAL}/kgeur_pca"
""")
plink.score(f"{LOCAL}/kgeur_pca.eigenvec.allele", f"{LOCAL}/part_in_kgeur",
            pfile=PANEL, freq=f"{LOCAL}/kgeur_pca.acount", extract=kg_ids,
            n_pcs=N_PCS_FIT, force=True)

part_k = plink.read_scores(f"{LOCAL}/part_in_kgeur.sscore", "person_id", N_PCS_FIT)
kg_k = pd.read_csv(f"{LOCAL}/kgeur_pca.eigenvec", sep=r"\s+")
kg_k = kg_k.rename(columns={("#IID" if "#IID" in kg_k.columns else "IID"): "sample"})
kg_k = kg_k.merge(refs.panel(), on="sample", how="left")

g_k = gate.fit(kg_k[kg_k["pop"].isin(refs.ANCHOR_POPS)], K_PCS, COVERAGE)
keep_kemper = pd.Series(g_k.keep(part_k), index=part_k["person_id"]).reindex(
    part["person_id"]).fillna(False).to_numpy()

sets = {"anchor (this run)": set(part.loc[keep, "person_id"]),
        "participant centroid": set(part.loc[keep_self, "person_id"]),
        "Kemper direction": set(part.loc[keep_kemper, "person_id"])}
d2 = f"{run.b}/01_ancestry_filtering/r2_eur_gbr_ceu_projection/aou_D2_keep_ids.txt"
if os.path.isfile(d2):
    s = pd.read_csv(d2, sep=r"\s+", header=None, dtype=str).iloc[:, -1]
    sets["eur_D2"] = set(s[s.str.isdigit()])

print("Jaccard overlap")
print(gate.overlap(sets, f"{OUT}/gate_overlap.tsv").round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, (i, j) in zip(axes, [(0, 1), (2, 3)]):
    a, b = f"PC{i+1}", f"PC{j+1}"
    ax.scatter(part[a], part[b], s=1, alpha=0.1, color="0.85", rasterized=True)
    for m, c, lab in ((keep_self & ~keep, "tab:orange", "participant centroid only"),
                      (keep & ~keep_self, "tab:blue", "anchor only")):
        ax.scatter(part[a][m], part[b][m], s=2, alpha=0.4, color=c, label=lab,
                   rasterized=True)
    for pop in refs.ANCHOR_POPS:
        s = kg[kg["pop"] == pop]
        ax.scatter(s[a], s[b], s=30, marker="x", color="k", zorder=3)
    if i < K_PCS and j < K_PCS:
        plots.ellipse(ax, g.mu, g.cov, i, j, g.threshold,
                      edgecolor="tab:blue", lw=1.4, zorder=4)
        plots.ellipse(ax, g_self.mu, g_self.cov, i, j, g_self.threshold,
                      edgecolor="tab:orange", lw=1.4, ls="--", zorder=4)
    ax.set_xlabel(a); ax.set_ylabel(b)
axes[0].legend(fontsize=8, markerscale=4)
fig.suptitle("who the anchor changes, against the participants' own centroid")
plt.tight_layout()
plt.savefig(f"{OUT}/gate_difference.png", dpi=130, bbox_inches="tight")
plt.show()

Next: `covariate_pcs.ipynb`.